# Variograms and Ordinary Kriging

This notebook simulates a stationary Gaussian random field, estimates an empirical semivariogram, fits a simple exponential model, and performs ordinary kriging.

The purpose is to expose the linear algebra rather than hide it behind a geostatistics package.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.spatial.distance import cdist, pdist

rng = np.random.default_rng(9)
n = 70
coords = rng.uniform(0, 1, size=(n, 2))
D = cdist(coords, coords)

process_variance = 0.8
correlation_range = 0.18
nugget_variance = 0.06

C = process_variance * np.exp(-D / correlation_range)
latent = rng.multivariate_normal(np.full(n, 6.5), C + 1e-10*np.eye(n))
z = latent + rng.normal(scale=np.sqrt(nugget_variance), size=n)


## Empirical semivariogram

For pairs in a lag bin \(N(h)\),

\[
\hat\gamma(h)=\frac{1}{2N(h)}\sum_{(i,j)\in N(h)}[Z(s_i)-Z(s_j)]^2.
\]


In [ ]:
distances = pdist(coords)
semivariance = 0.5 * pdist(z[:, None], metric="sqeuclidean")
max_distance = np.quantile(distances, 0.65)
edges = np.linspace(0, max_distance, 13)
centers = 0.5 * (edges[:-1] + edges[1:])

gamma_hat = np.full(len(centers), np.nan)
counts = np.zeros(len(centers), dtype=int)
for b in range(len(centers)):
    mask = (distances >= edges[b]) & (distances < edges[b+1])
    counts[b] = mask.sum()
    if counts[b]:
        gamma_hat[b] = semivariance[mask].mean()


In [ ]:
def exponential_gamma(h, nugget, partial_sill, a):
    h = np.asarray(h)
    g = nugget + partial_sill * (1 - np.exp(-h/a))
    return np.where(h == 0, 0.0, g)

valid = np.isfinite(gamma_hat)
x = centers[valid]
y = gamma_hat[valid]
w = np.sqrt(counts[valid])

scale = max(float(np.max(y)), 1e-6)
params, _ = curve_fit(
    exponential_gamma, x, y,
    p0=(0.05*scale, 0.95*scale, np.median(x)),
    bounds=((0, 1e-8, 1e-6), (3*scale, 5*scale, 10*np.max(x))),
    sigma=1/w, maxfev=20000
)
params


In [ ]:
h_grid = np.linspace(0, max_distance, 200)
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(centers, gamma_hat, s=np.maximum(counts/5, 10), label="empirical")
ax.plot(h_grid, exponential_gamma(h_grid, *params), label="fitted exponential")
ax.set(xlabel="distance", ylabel="semivariance", title="Empirical and fitted semivariogram")
ax.legend()
plt.show()


## Ordinary kriging system

With semivariogram matrix \(\Gamma\), ordinary kriging solves

\[
\begin{bmatrix}
\Gamma & \mathbf 1\\
\mathbf 1^\top & 0
\end{bmatrix}
\begin{bmatrix}
\lambda\\
\mu
\end{bmatrix}
=
\begin{bmatrix}
\gamma_0\\
1
\end{bmatrix}.
\]

With this sign convention, \(\sigma_K^2=\lambda^\top\gamma_0+\mu\).

In [ ]:
Gamma = exponential_gamma(D, *params)
system = np.block([
    [Gamma, np.ones((n, 1))],
    [np.ones((1, n)), np.zeros((1, 1))]
])

gx, gy = np.meshgrid(np.linspace(0, 1, 28), np.linspace(0, 1, 28))
targets = np.column_stack([gx.ravel(), gy.ravel()])
Dt = cdist(coords, targets)
Gamma_t = exponential_gamma(Dt, *params)
rhs = np.vstack([Gamma_t, np.ones((1, len(targets)))])

solution = np.linalg.solve(system, rhs)
weights = solution[:-1]
lagrange = solution[-1]
prediction = z @ weights
kriging_variance = np.sum(weights * Gamma_t, axis=0) + lagrange

pred_grid = prediction.reshape(gx.shape)
var_grid = kriging_variance.reshape(gx.shape)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
im0 = axes[0].imshow(pred_grid, origin="lower", extent=[0,1,0,1], aspect="auto")
axes[0].scatter(coords[:,0], coords[:,1], s=10)
axes[0].set_title("Ordinary-kriging prediction")
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(var_grid, origin="lower", extent=[0,1,0,1], aspect="auto")
axes[1].scatter(coords[:,0], coords[:,1], s=10)
axes[1].set_title("Kriging variance")
fig.colorbar(im1, ax=axes[1])
plt.show()


The uncertainty map depends strongly on sampling geometry. It is small where the fitted covariance model is supported by nearby observations and larger in gaps and near boundaries. It does **not** account for uncertainty from choosing the wrong mean or covariance model.